In [29]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import joblib
from datetime import datetime


In [38]:
data = pd.read_csv('inventory.csv')

# Convert 'created_at' to datetime and extract features
data['created_at'] = pd.to_datetime(data['created_at'])
data['month'] = data['created_at'].dt.month
data['day_of_week'] = data['created_at'].dt.dayofweek
data['year'] = data['created_at'].dt.year
data['days_since_start'] = (data['created_at'] - data['created_at'].min()).dt.days

# Define features and target
features = ['volume', 'month', 'day_of_week', 'year', 'days_since_start']
X = data[features]
y = data['price']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [40]:
# Train the model
model = LinearRegression()
model.fit(X_train, y_train)

# Evaluate the model
y_predictions = model.predict(X_test)
mse = mean_squared_error(y_test, y_predictions)
print(f'Mean Squared Error: {mse}')

# Save the model to a file
model_filename = 'fuel_price_model.joblib'
joblib.dump(model, model_filename)
print(f"Model saved to {model_filename}")


Mean Squared Error: 0.37218102875708453
Model saved to fuel_price_model.joblib


In [44]:
def predict_price(product, date):
    # Load the saved model
    model = joblib.load('fuel_price_model.joblib')

    # Prepare input data for the selected date
    input_data = pd.DataFrame({
        'created_at': [date]
    })

    # Add date-based features
    input_data['month'] = input_data['created_at'].dt.month
    input_data['day_of_week'] = input_data['created_at'].dt.dayofweek
    input_data['year'] = input_data['created_at'].dt.year
    input_data['days_since_start'] = (input_data['created_at'] - data['created_at'].min()).dt.days

    # Add other features (e.g., volume)
    # For simplicity, assume constant volume or use a forecast for volume
    input_data['volume'] = 1000  # Example: Assume constant volume

    # Define features
    features = ['volume', 'month', 'day_of_week', 'year', 'days_since_start']

    # Make prediction
    predicted_price = model.predict(input_data[features])
    return predicted_price[0]

In [51]:
product = 'petrol'  # User selects product
date = pd.to_datetime('2024-07-15')  # User selects date
predicted_price = predict_price(product, date)
print(f"Predicted price for {product} on {date}: {predicted_price}")

Predicted price for petrol on 2024-07-15 00:00:00: 0.9733913256673077
